# Qiskit Runtime V2 primitives



To reduce the total job execution time, V2 primitives only accept circuits and observables that use instructions supported by the target QPU (quantum processing unit). Such circuits and observables are referred to as instruction set architecture (ISA) circuits and observables. V2 primitives do not perform layout, routing, and translation operations.

#### Estimator V2

The new interface lets you specify a single circuit and multiple observables and parameter value sets for that circuit, so that sweeps over parameter value sets and observables can be efficiently specified. Previously, you had to specify the same circuit multiple times to match the size of the data to be combined. Also, while you can still use resilience_level as the simple knob, V2 primitives give you the flexibility to turn on or off individual error mitigation / suppression methods to customize them for your needs.

#### SamplerV2

It's a class for interacting with Qiskit Runtime Sampler primitive service. This class supports version 2 of the Sampler interface, which uses different input and output formats than version 1. Qiskit Runtime Sampler primitive returns the sampled result according to the specified output type. For example, it returns a bitstring for each shot if measurement level 2 (bits) is requested. The run() method can be used to submit circuits and parameters to the Sampler primitive. It initializes the Sampler primitive. It is simplified to focus on its core task of sampling the output register from execution of quantum circuits. It returns the samples, whose type is defined by the program, without weights. The output data is also separated by the output register names defined by the program. This change enables future support for circuits with classical control flow.

#### Changes from V1 to V2

In [4]:
# import 

from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import SamplerV2 as Sampler

Both SamplerV2 and EstimatorV2 take one or more primitive unified blocs (PUBs) as the input. Each PUB is a tuple that contains one circuit and the data broadcasted to that circuit, which can be multiple observables and parameters. Each PUB returns a result.

* Sampler V2 PUB format: (<circuit>, <parameter values>, <shots>), where <parameter values>and <shots> are optional.
* Estimator V2 PUB format: (<circuit>, <observables>, <parameter values>, <precision>), where <parameter values>and <precision> are optional.   Numpy broadcasting rules are used when combining observables and parameter values.

Additionally, the following changes have been made:
Estimator V2 has gained a precision argument in the run() method that specifies the targeted precision of the expectation value estimates.
Sampler V2 has the shots argument in its run() method.

In [23]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
circuit1 = QuantumCircuit(2)
circuit1.cx(0, 1)
circuit2 = QuantumCircuit(2)
circuit2.x(0)
observable = SparsePauliOp.from_list([("ZZ", 1), ("XX", 0.5)])

# Define observable arrays for the PUBs
obs_array1 = [observable]
obs_array_2 = [observable]

In [11]:
!pip install qiskit-ibm-runtime --upgrade

In [24]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
sampler = Sampler(backend)

In [25]:
# Estimate expectation values for two PUBs, both with 0.05 precision.
estimator.run([(circuit1, obs_array1), (circuit2, obs_array_2)], precision=0.05)

IBMInputValueError: 'The instruction cx on qubits (0, 1) is not supported by the target system. Circuits that do not match the target hardware definition are no longer supported after March 4, 2024. See the transpilation documentation (https://quantum.cloud.ibm.com/docs/guides/transpile) for instructions to transform circuits and the primitive examples (https://quantum.cloud.ibm.com/docs/guides/primitives-examples) to see this coupled with operator transformations.'

In [7]:
# Sample two circuits at 128 shots each.
sampler.run([circuit1, circuit2], shots=128)

# Sample two circuits at different amounts of shots.
# The "None"s are necessary as placeholders
# for the lack of parameter values in this example.
sampler.run([
  (circuit1, None, 123),
  (circuit2, None, 456),
])

NameError: name 'sampler' is not defined

The output is now in the PubResult format. A PubResult is the data and metadata resulting from a single PUB’s execution.

Estimator V2 continues to return expectation values.

The data portion of a Estimator V2 PubResult contains both expectation values and standard errors (stds). V1 returned variance in metadata.

Sampler V2 returns per-shot measurements in the form of bitstrings, instead of the quasi-probability distributions from the V1 interface. The bitstrings show the measurement outcomes, preserving the shot order in which they were measured.

Sampler V2 has convenience methods like get_counts() to help with migration.

The Sampler V2 result objects organize data in terms of their input circuits' classical register names, for compatibility with dynamic circuits. By default, the classical register name is meas, as shown in the following example. When defining your circuit, if you create one or more classical registers with a non-default name, use that name to get the results. You can find the classical register name by running <circuit_name>.cregs. For example, qc.cregs.

In [ ]:
# Define a quantum circuit with 2 qubits
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure_all()
circuit.draw()

NameError: name 'QuantumCircuit' is not defined

In [ ]:
# Estimator V1: Execute 1 circuit with 4 observables
job = estimator_v1.run([circuit] * 4, [obs1, obs2, obs3, obs4])
evs = job.result().values

# Estimator V2: Execute 1 circuit with 4 observables
job = estimator_v2.run([(circuit, [obs1, obs2, obs3, obs4])])
evs = job.result()[0].data.evs

NameError: name 'estimator_v1' is not defined

In [ ]:
# Estimator V1: Execute 1 circuit with 4 observables and 2 parameter sets
job = estimator_v1.run([circuit] * 8, [obs1, obs2, obs3, obs4] * 2, [vals1, vals2] * 4)
evs = job.result().values

# Estimator V2: Execute 1 circuit with 4 observables and 2 parameter sets

job = estimator_v2.run([(circuit, [[obs1], [obs2], [obs3], [obs4]], [[vals1], [vals2]])])
evs = job.result()[0].data.evs

NameError: name 'estimator_v1' is not defined

In [ ]:
# Estimator V1: Cannot execute 2 circuits with different observables

# Estimator V2: Execute 2 circuits with 2 different observables.  There are
# two PUBs because each PUB can have only one circuit.
job = estimator_v2.run([(circuit1, obs1), (circuit2, obs2)])
evs1 = job.result()[0].data.evs  # result for pub 1 (circuit 1)
evs2 = job.result()[1].data.evs  # result for pub 2 (circuit 2)

NameError: name 'estimator_v2' is not defined

In [ ]:
  # Sampler V1: Execute 1 circuit with 3 parameter sets
  job = sampler_v1.run([circuit] * 3, [vals1, vals2, vals3])
  dists = job.result().quasi_dists

  # Sampler V2: Executing 1 circuit with 3 parameter sets
  job = sampler_v2.run([(circuit, [vals1, vals2, vals3])])
  counts = job.result()[0].data.meas.get_counts()

In [ ]:
# Sampler V1: Execute 2 circuits with 1 parameter set
  job = sampler_v1.run([circuit1, circuit2], [vals1] * 2)
  dists = job.result().quasi_dists

  # Sampler V2: Execute 2 circuits with 1 parameter set
  job = sampler_v2.run([(circuit1, vals1), (circuit2, vals1)])
  counts1 = job.result()[0].data.meas.get_counts()  # result for pub 1 (circuit 1)
  counts2 = job.result()[1].data.meas.get_counts()  # result for pub 2 (circuit 2)

In [ ]:
v2_result = sampler_v2_job.result()
v1_format = []
for pub_result in v2_result:
    counts = pub_result.data.meas.get_counts()
    v1_format.append( {int(key, 2): val/shots for key, val in counts.items()} )

In [ ]:
from qiskit import ClassicalRegister, QuantumRegister, QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

alpha = ClassicalRegister(5, "alpha")
beta = ClassicalRegister(7, "beta")
qreg = QuantumRegister(12)

circuit = QuantumCircuit(qreg, alpha, beta)
circuit.h(0)
circuit.measure(qreg[:5], alpha)
circuit.measure(qreg[5:], beta)

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False, min_num_qubits=12)
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(circuit)

sampler = Sampler(backend)
job = sampler.run([isa_circuit])
result = job.result()
# Get results for the first (and only) PUB
pub_result = result[0]
print(f" >> Counts for the alpha output register: {pub_result.data.alpha.get_counts()}")
print(f" >> Counts for the beta output register: {pub_result.data.beta.get_counts()}")

Options are specified differently in the V2 primitives in these ways:

* SamplerV2 and EstimatorV2 now have separate options classes. You can see the available options and update option values during or after primitive initialization.
* Instead of the set_options() method, V2 primitive options have the update() method that applies changes to the options attribute.
* If you do not specify a value for an option, it is given a special value of Unset and the server defaults are used.
* For V2 primitives, the options attribute is the dataclass Python type. You can use the built-in asdict method to convert it to a dictionary.

In [ ]:
from dataclasses import asdict
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

# Setting options after primitive initialization
# This uses auto complete.
estimator.options.default_shots = 4000
# This does bulk update.
estimator.options.update(default_shots=4000, resilience_level=2)

# Print the dictionary format.
# Server defaults are used for unset options.
print(asdict(estimator.options))

{'_VERSION': 2, 'max_execution_time': Unset, 'environment': {'log_level': 'WARNING', 'job_tags': None, 'private': False}, 'simulator': {'noise_model': Unset, 'seed_simulator': Unset, 'coupling_map': Unset, 'basis_gates': Unset}, 'default_precision': Unset, 'default_shots': 4000, 'resilience_level': 2, 'seed_estimator': Unset, 'dynamical_decoupling': {'enable': Unset, 'sequence_type': Unset, 'extra_slack_distribution': Unset, 'scheduling_method': Unset, 'skip_reset_qubits': Unset}, 'resilience': {'measure_mitigation': Unset, 'measure_noise_learning': {'num_randomizations': Unset, 'shots_per_randomization': Unset}, 'zne_mitigation': Unset, 'zne': {'amplifier': Unset, 'noise_factors': Unset, 'extrapolator': Unset, 'extrapolated_noise_factors': Unset}, 'pec_mitigation': Unset, 'pec': {'max_overhead': Unset, 'noise_gain': Unset}, 'layer_noise_learning': {'max_layers_to_learn': Unset, 'shots_per_randomization': Unset, 'num_randomizations': Unset, 'layer_pair_depths': Unset}, 'layer_noise_mod

In [ ]:
from dataclasses import asdict
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
sampler = Sampler(backend, options={"default_shots": 4096})

# Setting options after primitive initialization
# This uses auto complete.
sampler.options.dynamical_decoupling.enable = True
# Turn on gate twirling. Requires qiskit_ibm_runtime 0.23.0 or later.
sampler.options.twirling.enable_gates = True

# This does bulk update.  The value for default_shots is overridden if you specify shots with run() or in the PUB.
sampler.options.update(default_shots=1024, dynamical_decoupling={"sequence_type": "XpXm"})

# Print the dictionary format.
# Server defaults are used for unset options.
print(asdict(sampler.options))

{'_VERSION': 2, 'max_execution_time': Unset, 'environment': {'log_level': 'WARNING', 'job_tags': None, 'private': False}, 'simulator': {'noise_model': Unset, 'seed_simulator': Unset, 'coupling_map': Unset, 'basis_gates': Unset}, 'default_shots': 1024, 'dynamical_decoupling': {'enable': True, 'sequence_type': 'XpXm', 'extra_slack_distribution': Unset, 'scheduling_method': Unset, 'skip_reset_qubits': Unset}, 'execution': {'init_qubits': Unset, 'rep_delay': Unset, 'meas_type': Unset}, 'twirling': {'enable_gates': True, 'enable_measure': Unset, 'num_randomizations': Unset, 'shots_per_randomization': Unset, 'strategy': Unset}, 'experimental': Unset}


#### Error mitigation and suppression

* Because Sampler V2 returns samples without postprocessing, it does not support resilience levels.

* Sampler V2 does not support optimization_level.

* Estimator V2 will drop support for optimization_level on or around 30 September 2024.

* Estimator V2 does not support resilience level 3. This is because resilience level 3 in V1 Estimator uses Probabilistic Error Cancellation (PEC), which is proven to give unbiased results at the cost of exponential processing time. Level 3 was removed to draw attention to that tradeoff. You can, however, still use PEC as the error mitigation method by specifying the pec_mitigation option.

* Estimator V2 supports resilience_level 0-2, as described in the following table. These options are more advanced than their V1 counterparts. You can also explicitly turn on / off individual error mitigation / suppression methods.

In [23]:
from dataclasses import asdict
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend)

# Set resilience_level to 0
estimator.options.resilience_level = 0

# Turn on measurement error mitigation
estimator.options.resilience.measure_mitigation = True

In [24]:
from qiskit_ibm_runtime import SamplerV2 as Sampler

sampler = Sampler(backend)
# Turn on dynamical decoupling with sequence XpXm.
sampler.options.dynamical_decoupling.enable = True
sampler.options.dynamical_decoupling.sequence_type = "XpXm"

print(f">> dynamical decoupling sequence to use: {sampler.options.dynamical_decoupling.sequence_type}")

>> dynamical decoupling sequence to use: XpXm


#### Transpilation
V2 primitives support only circuits that adhere to the Instruction Set Architecture (ISA) of a particular backend. Because the primitives do not perform layout, routing, and translation operations, the corresponding transpilation options from V1 are not supported.

#### Job status
The V2 primitives have a new RuntimeJobV2 class, which inherits from BasePrimitiveJob. The status() method of this new class returns a string instead of a JobStatus enum from Qiskit.

In [25]:
job = estimator.run(...)

# check if a job is still running
print(f"Job {job.job_id()} is still running: {job.status() == "RUNNING"}")

TypeError: 'ellipsis' object is not iterable

Steps to migrate to Estimator V2
Replace from qiskit_ibm_runtime import Estimator with from qiskit_ibm_runtime import EstimatorV2 as Estimator.

Remove any from qiskit_ibm_runtime import Options statements, since the Options class is not used by V2 primitives. You can instead pass options as a dictionary when initializing the EstimatorV2 class (for example estimator = Estimator(backend, options={“dynamical_decoupling”: {“enable”: True}})), or set them after initialization:


estimator = Estimator(backend)
estimator.options.dynamical_decoupling.enable = True
Review all the supported options and make updates accordingly.

Group each circuit you want to run with the observables and parameter values you want to apply to the circuit in a tuple (a PUB). For example, use (circuit1, observable1, parameter_set1) if you want to run circuit1 with observable1 and parameter_set1.

You might need to reshape your arrays of observables or parameter sets if you want to apply their outer product. For example, an array of observables of shape (4, 1) and an array of parameter sets of shape (1, 6) will give you a result of (4, 6) expectation values. See the Numpy broadcasting rules for more details.

You can optionally specify the precision you want for that specific PUB.

Update the estimator run() method to pass in the list of PUBs. For example, run([(circuit1, observable1, parameter_set1)]). You can optionally specify a precision here, which would apply to all PUBs.

Estimator V2 job results are grouped by PUBs. You can see the expectation value and standard error for each PUB by indexing to it. For example:


pub_result = job.result()[0]
print(f">>> Expectation values: {pub_result.data.evs}")
print(f">>> Standard errors: {pub_result.data.stds}")

Run a single experiment
Use Estimator to determine the expectation value of a single circuit-observable pair.

In [26]:
import numpy as np
from qiskit.circuit.library import IQP
from qiskit.quantum_info import SparsePauliOp, random_hermitian
from qiskit_ibm_runtime import EstimatorV2 as Estimator, QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False, min_num_qubits=127)
estimator = Estimator(backend)

n_qubits = 127

mat = np.real(random_hermitian(n_qubits, seed=1234))
circuit = IQP(mat)
observable = SparsePauliOp("Z" * n_qubits)

pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
isa_circuit = pm.run(circuit)
isa_observable = observable.apply_layout(isa_circuit.layout)

job = estimator.run([(isa_circuit, isa_observable)])
result = job.result()

print(f" > Expectation value: {result[0].data.evs}")
print(f" > Metadata: {result[0].metadata}")

C:\Users\nikhi\AppData\Local\Temp\ipykernel_53108\4255980472.py:13: DeprecationWarning: The class ``qiskit.circuit.library.iqp.IQP`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the qiskit.circuit.library.iqp function instead.
  circuit = IQP(mat)


 > Expectation value: -0.9739130434782609
 > Metadata: {'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32}


Run multiple experiments in a single job
Use Estimator to determine the expectation values of multiple circuit-observable pairs.

In [27]:
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)

n_qubits = 3
rng = np.random.default_rng()
mats = [np.real(random_hermitian(n_qubits, seed=rng)) for _ in range(3)]
circuits = [IQP(mat) for mat in mats]
observables = [
    SparsePauliOp("X" * n_qubits),
    SparsePauliOp("Y" * n_qubits),
    SparsePauliOp("Z" * n_qubits),
]

isa_circuits = pm.run(circuits)
isa_observables = [ob.apply_layout(isa_circuits[0].layout) for ob in observables]


estimator = Estimator(backend)
job = estimator.run([(isa_circuits[0], isa_observables[0]),(isa_circuits[1], isa_observables[1]),(isa_circuits[2], isa_observables[2])])
job_result = job.result()
for idx in range(len(job_result)):
    pub_result = job_result[idx]
    print(f">>> Expectation values for PUB {idx}: {pub_result.data.evs}")
    print(f">>> Standard errors for PUB {idx}: {pub_result.data.stds}")

C:\Users\nikhi\AppData\Local\Temp\ipykernel_53108\882246781.py:9: DeprecationWarning: The class ``qiskit.circuit.library.iqp.IQP`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the qiskit.circuit.library.iqp function instead.
  circuits = [IQP(mat) for mat in mats]


>>> Expectation values for PUB 0: 0.012363098837599945
>>> Standard errors for PUB 0: 0.019473524305623612
>>> Expectation values for PUB 1: -0.014513202983269502
>>> Standard errors for PUB 1: 0.013081041101404614
>>> Expectation values for PUB 2: 0.711684472216623
>>> Standard errors for PUB 2: 0.011564117212177349


Run parameterized circuits
Use Estimator to run multiple experiments in a single job, leveraging parameter values to increase circuit reusability. In the following example, notice that steps 1 and 2 are the same for V1 and V2.

In [28]:
import numpy as np

from qiskit.circuit import QuantumCircuit, Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService

# Step 1: Map classical inputs to a quantum problem

theta = Parameter("θ")

chsh_circuit = QuantumCircuit(2)
chsh_circuit.h(0)
chsh_circuit.cx(0, 1)
chsh_circuit.ry(theta, 0)

number_of_phases = 21
phases = np.linspace(0, 2 * np.pi, number_of_phases)
individual_phases = [[ph] for ph in phases]

ZZ = SparsePauliOp.from_list([("ZZ", 1)])
ZX = SparsePauliOp.from_list([("ZX", 1)])
XZ = SparsePauliOp.from_list([("XZ", 1)])
XX = SparsePauliOp.from_list([("XX", 1)])
ops = [ZZ, ZX, XZ, XX]

# Step 2: Optimize problem for quantum execution.

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
chsh_isa_circuit = pm.run(chsh_circuit)
isa_observables = [operator.apply_layout(chsh_isa_circuit.layout) for operator in ops]

from qiskit_ibm_runtime import EstimatorV2 as Estimator

# Step 3: Execute using Qiskit primitives.

# Reshape observable array for broadcasting
reshaped_ops = np.fromiter(isa_observables, dtype=object)
reshaped_ops = reshaped_ops.reshape((4, 1))

estimator = Estimator(backend, options={"default_shots": int(1e4)})
job = estimator.run([(chsh_isa_circuit, reshaped_ops, individual_phases)])
# Get results for the first (and only) PUB
pub_result = job.result()[0]
print(f">>> Expectation values: {pub_result.data.evs}")
print(f">>> Standard errors: {pub_result.data.stds}")
print(f">>> Metadata: {pub_result.metadata}")

>>> Expectation values: [[ 0.94459932  0.89956685  0.7605724   0.5390906   0.27344234 -0.02727928
  -0.3082992  -0.56983392 -0.77551106 -0.91060846 -0.95325941 -0.89047375
  -0.74281922 -0.54060612 -0.27387535  0.0197017   0.33579499  0.56853491
   0.7984603   0.91558801  0.95607393]
 [ 0.02684628  0.30526817  0.57611249  0.77291303  0.90779393  0.94005277
   0.89263878  0.76252092  0.53064701  0.27041131 -0.03074332 -0.30613418
  -0.57546298 -0.79348075 -0.91363949 -0.94719734 -0.90194837 -0.73783967
  -0.53800809 -0.27387535  0.03788789]
 [-0.02533076 -0.34012503 -0.59949473 -0.8086359  -0.93485671 -0.98616775
  -0.93550622 -0.79499626 -0.55662728 -0.29314405  0.04243444  0.34055804
   0.60577329  0.81686299  0.9433003   0.98400272  0.9240316   0.77572756
   0.55294675  0.27235983 -0.02229973]
 [ 0.98421923  0.92186658  0.77074801  0.54580217  0.2795044  -0.04763049
  -0.33319696 -0.5774115  -0.82184254 -0.94156828 -0.98443573 -0.93269169
  -0.77659357 -0.57632899 -0.27474135  0.0259

Steps to migrate to Sampler V2
Replace from qiskit_ibm_runtime import Sampler with from qiskit_ibm_runtime import SamplerV2 as Sampler.
Remove any from qiskit_ibm_runtime import Options statements, since the Options class is not used by V2 primitives. You can instead pass options as a dictionary when initializing the SamplerV2 class (for example sampler = Sampler(backend, options={“default_shots”: 1024})), or set them after initialization:

sampler = Sampler(backend)
sampler.options.default_shots = 1024
Review all the supported options and make updates accordingly.
Group each circuit you want to run with the observables and parameter values you want to apply to the circuit in a tuple (a PUB). For example, use (circuit1, parameter_set1) if you want to run circuit1 with parameter_set1. You can optionally specify the shots you want for that specific PUB.
Update the sampler run() method to pass in the list of PUBs. For example, run([(circuit1, parameter_set1)]). You can optionally specify shots here, which would apply to all PUBs.
Sampler V2 job results are grouped by PUBs. You can see the output data for each PUB by indexing to it. While Sampler V2 returns unweighted samples, the result class has a convenience method to get counts instead. For example:

pub_result = job.result()[0]
print(f">>> Counts: {pub_result.data.meas.get_counts()}")
print(f">>> Per-shot measurement: {pub_result.data.meas.get_counts()}")

Run a single experiment
Use Sampler to determine the counts or quasi-probability distribution of a single circu

In [29]:
import numpy as np
from qiskit.circuit.library import IQP
from qiskit.quantum_info import random_hermitian
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

service = QiskitRuntimeService()

backend = service.least_busy(operational=True, simulator=False, min_num_qubits=127)

n_qubits = 127

mat = np.real(random_hermitian(n_qubits, seed=1234))
circuit = IQP(mat)
circuit.measure_all()

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(circuit)

sampler = Sampler(backend)
job = sampler.run([isa_circuit])
result = job.result()

C:\Users\nikhi\AppData\Local\Temp\ipykernel_53108\2401533443.py:14: DeprecationWarning: The class ``qiskit.circuit.library.iqp.IQP`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the qiskit.circuit.library.iqp function instead.
  circuit = IQP(mat)
c:\Users\nikhi\anaconda3\Lib\site-packages\qiskit_ibm_runtime\qiskit_runtime_service.py:1212: UserWarning: This instance has met its usage limit. Workloads will not run until time is made available. Check https://quantum.cloud.ibm.com/instances/crn%3Av1%3Abluemix%3Apublic%3Aquantum-computing%3Aus-east%3Aa%2F948cd7f1597b4c6db2c54aead43b2b46%3A2b89e4d4-e47e-4a99-9c0b-e7170c20cdf6%3A%3A for more details.
  warnings.warn(


Run multiple experiments in a single job
Use Sampler to determine the counts or quasi-probability distributions of multiple circuits in one job.

In [30]:
import numpy as np
from qiskit.circuit.library import IQP
from qiskit.quantum_info import random_hermitian
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

service = QiskitRuntimeService()

backend = service.least_busy(operational=True, simulator=False, min_num_qubits=127)

n_qubits = 127

rng = np.random.default_rng()
mats = [np.real(random_hermitian(n_qubits, seed=rng)) for _ in range(3)]
circuits = [IQP(mat) for mat in mats]
for circuit in circuits:
    circuit.measure_all()

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuits = pm.run(circuits)

sampler = Sampler(backend)
job = sampler.run(isa_circuits)
result = job.result()

for idx, pub_result in enumerate(result):
    print(f" > Counts for pub {idx}: {pub_result.data.meas.get_counts()}")

C:\Users\nikhi\AppData\Local\Temp\ipykernel_53108\534761376.py:14: DeprecationWarning: The class ``qiskit.circuit.library.iqp.IQP`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the qiskit.circuit.library.iqp function instead.
  circuits = [IQP(mat) for mat in mats]


KeyboardInterrupt: 

Run parameterized circuits
Run several experiments in a single job, leveraging parameter values to increase circuit reusability.

In [ ]:
import numpy as np
from qiskit.circuit.library import RealAmplitudes
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService

# Step 1: Map classical inputs to a quantum problem
num_qubits = 127
circuit = RealAmplitudes(num_qubits=num_qubits, reps=2)
circuit.measure_all()

# Define three sets of parameters for the circuit
rng = np.random.default_rng(1234)
parameter_values = [
    rng.uniform(-np.pi, np.pi, size=circuit.num_parameters) for _ in range(3)
]

# Step 2: Optimize problem for quantum execution.

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False, min_num_qubits=num_qubits)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(circuit)

# Step 3: Execute using Qiskit primitives.

from qiskit_ibm_runtime import SamplerV2 as Sampler

sampler = Sampler(backend)
job = sampler.run([(isa_circuit, parameter_values)])
result = job.result()
# Get results for the first (and only) PUB
pub_result = result[0]
# Get counts from the classical register "meas".
print(f" >> Counts for the meas output register: {pub_result.data.meas.get_counts()}")